# Notebook 12 — HDC-RWKV: Gradient-Trainable Bipolar Recurrence

*The novel architectural contribution. Combines HDC stability with gradient-trained adaptability.*

## What we're doing and why

### The honest distillation

- **Vanilla HDC (nb11)**: stable, hardware-perfect, but **non-adaptive** — vocab is random, training is one-pass Hebbian. Quality ceiling is low.
- **Vanilla RWKV (nb10)**: adaptive via gradients, recurrent, but uses floating-point operations that can produce NaN, overflow, and unbounded state magnitudes.

This notebook builds the hybrid: a recurrent architecture where **every learnable parameter is bipolar at inference**, but training uses gradient descent through the **straight-through estimator** (STE).

### The architectural shape

```
token_t at position t
  │
  ▼
learned vocab_hv[token_t] (continuous in training; bipolar at inference)
  │
  ⊗ permute_t(·)             ← position-dependent binding (rotation)
  ▼
input_hv_t (bipolar)
  │
  ▼ RECURRENCE:
     state_t = sign( decay_mask ⊙ state_{t-1} + input_hv_t )
            ↑ element-wise multiply, then add, then sign
            ↑ state stays in {-1,+1}^d at every step
            ↑ no exp, no softmax, no NaN possible
  ▼
state_t (bipolar) → similarity to each prototype_hv[v] → softmax → next-token prob
```

### Five things that are different from anything before

1. **The recurrent state is bipolar at every step**. Not float, not int8, but {-1, +1}^d. After every recurrence step we take `sign()`.
2. **decay_mask is a per-channel bipolar pattern**. Each channel can be `+1` (keep state contribution) or `-1` (flip it). Learned via gradient descent through STE.
3. **Vocab and prototypes are gradient-learned hypervectors**. Unlike nb11's random vocab, these are placed by training in semantically-meaningful positions.
4. **Output via prototype similarity, not logits-projection**. Closer to retrieval than generation. Bounds output to learned geometry.
5. **Single Hebbian-like update during training, plus gradient flow via STE**. The combination is genuinely novel for autoregressive LMs.

### Honest scientific posture

We do not know yet if this beats RWKV at our scale. The architecture has theoretical appeal (stability + adaptability) but no published precedent at sub-32KB autoregressive LMs. This notebook is the first empirical test.


## Cell 1 — Setup + BPE tokenizer (same as nb07b/nb10/nb11)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, struct
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

torch.manual_seed(1337)
np.random.seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

text = Path('../data/tinyshakespeare.txt').read_text().lower()

def train_bpe(text, num_merges):
    EOW = '</w>'
    word_freq = Counter(tuple(list(w) + [EOW]) for w in text.split())
    word_lists = {w: list(w) for w in word_freq}
    merges = []
    for _ in range(num_merges):
        pair_counts = Counter()
        for w, freq in word_freq.items():
            sym = word_lists[w]
            for i in range(len(sym)-1):
                pair_counts[(sym[i], sym[i+1])] += freq
        if not pair_counts: break
        best, _ = pair_counts.most_common(1)[0]
        new_tok = best[0] + best[1]
        merges.append((best, new_tok))
        for w in word_freq:
            sym = word_lists[w]; new_sym = []; i = 0
            while i < len(sym):
                if i < len(sym)-1 and (sym[i], sym[i+1]) == best:
                    new_sym.append(new_tok); i += 2
                else:
                    new_sym.append(sym[i]); i += 1
            word_lists[w] = new_sym
    vocab_set = set()
    for w in word_freq:
        vocab_set.update(word_lists[w]); vocab_set.update(w)
    return merges, sorted(vocab_set)

EOW = '</w>'
VOCAB_SIZE = 128
merges, vocab = train_bpe(text, 88)
all_toks = ['<unk>'] + sorted(vocab)
while len(all_toks) < VOCAB_SIZE: all_toks.append(f'<pad{len(all_toks)}>')
all_toks = all_toks[:VOCAB_SIZE]
itos = all_toks
stoi = {t: i for i, t in enumerate(itos)}

def encode_word(word):
    sym = list(word) + [EOW]
    for (a, b), m in merges:
        i = 0; new_sym = []
        while i < len(sym):
            if i < len(sym)-1 and sym[i]==a and sym[i+1]==b:
                new_sym.append(m); i += 2
            else:
                new_sym.append(sym[i]); i += 1
        sym = new_sym
    return [stoi.get(s, 0) for s in sym]

def encode(text):
    out = []
    for w in text.split():
        out.extend(encode_word(w))
    return out
def decode(ids):
    return ''.join(itos[i] for i in ids).replace(EOW, ' ')

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f'tokens: train {len(train_data):,}  val {len(val_data):,}')


## Cell 2 — The Straight-Through Estimator (STE) — the trick that makes this trainable

### The problem

`sign(x)` is non-differentiable: its derivative is the Dirac delta at 0 and zero everywhere else. You can't train through it with standard backprop.

### The solution (Bengio 2013, Courbariaux 2016)

Use `sign(x)` in the forward pass, but pretend the gradient through it is the identity function in the backward pass. This is a *biased* gradient estimator, but it works remarkably well in practice and is the standard way to train binary neural networks.

```python
def ste_sign(x):
    # Forward: sign(x). Backward: gradient passes through as if it were x (or clamped x).
    return x.sign() + (x - x.detach())
```

The `(x - x.detach())` is a clever PyTorch idiom: in the forward pass it equals 0 (so the output is `sign(x)`); in the backward pass it contributes the gradient of `x` (so the gradient flows through). A small refinement is to clamp `x` to `[-1, +1]` in the backward path — gradients only flow when `x` is in the 'sign-near-zero' regime.

### What this lets us do

Train a model where the deployed parameters are bipolar bits — using standard PyTorch optimizers. Sign-quantization happens implicitly during the forward pass; gradients flow normally; parameters get pushed toward the bipolar-friendly geometry that minimizes loss.


In [ ]:
def ste_sign(x):
    """Sign function with straight-through gradient estimator.
    Forward: sign(x). Backward: gradient passes through clamped x."""
    return x.sign().detach() + x.clamp(-1, 1) - x.clamp(-1, 1).detach()

# Sanity check: forward returns sign, backward returns 1 (or 0 outside clamp)
x = torch.tensor([0.3, -0.7, 1.5, -2.1], requires_grad=True)
y = ste_sign(x)
loss = y.sum()
loss.backward()
print(f'x:    {x.tolist()}')
print(f'sign: {y.tolist()}  (should be +1/-1 each)')
print(f'grad: {x.grad.tolist()}  (should be 1 inside [-1,1], 0 outside)')


## Cell 3 — Hyperparameters

Budget for fitting in 32 KB EEPROM:

- `d = 512` — hypervector dimension. 64 bytes per bipolar vector. Smaller than nb11's 1024 because we have extra structures.
- `vocab = 128`, `T = 16` (max context window)
- `n_layers = 1` — one recurrent layer is the test; can stack later.
- No channel mixing in v1 — keep it simple, validate the core hypothesis.

### Storage budget at inference (deployment)

| Component | Shape | Bits | Bytes |
|---|---|---|---|
| vocab_hv | (V, d) | 128 × 512 | 8,192 |
| prototype_hv | (V, d) | 128 × 512 | 8,192 |
| decay_mask | (d,) | 512 | 64 |
| **Total weights** | | | **16,448** |

Plus BPE tokenizer table (~5 KB on Arduino flash, not in EEPROM) and headers. **EEPROM target: ~17 KB** — we have generous headroom.

### Training-time storage

Training uses continuous (float32) parameters of the same shapes, materialized as bipolar via `ste_sign()` in each forward pass. Total trainable params: ~131K floats during training — still tiny.


In [ ]:
BATCH_SIZE = 64
BLOCK_SIZE = 16     # training context length
D          = 512    # hypervector dimension
N_LAYERS   = 1      # REVERTED from 2 — depth stacking hurt; 1-layer was the best HDC-RWKV result
LR         = 3e-3
N_STEPS    = 12000
EVAL_EVERY = 500

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([d[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)


## Cell 4 — The HDC-RWKV architecture

### The recurrence

```
state_t = ste_sign( decay_mask ⊙ state_{t-1} + permute_t(vocab_hv[token_t]) )
```

Three operations per token:

1. **Element-wise multiply** the previous state with `decay_mask`. Each channel either keeps its sign (`+1`) or flips it (`-1`).
2. **Add** the input contribution: the current token's bipolar vector, rotated by `t` positions.
3. **Take sign()** with STE — the state stays in {-1, +1}^d.

This is the simplest bipolar recurrence we can write down. The state is **always bipolar**. The decay mask provides per-channel memory dynamics. The position-rotation gives temporal ordering.

### Why no MLP / channel mixing yet?

Adding more layers complicates the empirical test. We're checking the *core hypothesis*: can bipolar gradient-trained recurrence learn useful representations for autoregressive LM? If yes, we extend in nb12b/12c. If no, simplifying further (or adding it) is the next step.

### Output: prototype similarity

Final hidden state of the recurrence is compared to learned prototype hypervectors (one per vocab token) via dot product. Softmax of similarities gives next-token distribution. Same as nb11's HDC but with the prototypes also being gradient-trained.


In [ ]:
class HDCRWKV(nn.Module):
    """Bipolar recurrent LM with N stacked layers + residual + re-binarization.

    Each layer has its OWN learnable decay_mask.
    Between layers we add a RESIDUAL and re-BINARIZE via ste_sign — this is the
    Binary Neural Network (Courbariaux 2016) trick that makes stacked binary
    architectures actually train. Without it, layer 2 receives continuous tanh
    values it wasn't designed for and the model degrades.
    """
    def __init__(self, vocab_size, d, block_size, n_layers=2):
        super().__init__()
        self.vocab_size = vocab_size
        self.d          = d
        self.block_size = block_size
        self.n_layers   = n_layers

        self.vocab_hv_c     = nn.Parameter(torch.randn(vocab_size, d) * 0.5)
        self.prototype_hv_c = nn.Parameter(torch.randn(vocab_size, d) * 0.5)

        # One decay mask per layer
        self.decay_masks_c = nn.ParameterList([
            nn.Parameter(torch.full((d,), 0.5)) for _ in range(n_layers)
        ])

        self.log_temp = nn.Parameter(torch.tensor(math.log(math.sqrt(d))))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device = idx.device

        vocab_bp = ste_sign(self.vocab_hv_c)
        proto_bp = ste_sign(self.prototype_hv_c)

        tok_hv = vocab_bp[idx]
        rotated = torch.zeros_like(tok_hv)
        for t in range(T):
            rotated[:, t, :] = torch.roll(tok_hv[:, t, :], shifts=t, dims=-1)

        layer_input = rotated                        # (B, T, d) — bipolar
        for layer_idx in range(self.n_layers):
            decay_bp = ste_sign(self.decay_masks_c[layer_idx])
            state = torch.zeros(B, self.d, device=device)
            states = []
            for t in range(T):
                update = decay_bp * state + layer_input[:, t, :]
                state = torch.tanh(update)
                states.append(state)
            states = torch.stack(states, dim=1)      # (B, T, d), continuous

            # RESIDUAL + RE-BINARIZE: keeps layer 0's bipolar signal alive AND
            # ensures layer N+1 receives bipolar input matching its design.
            if layer_idx < self.n_layers - 1:
                layer_input = ste_sign(states + layer_input)
            else:
                # Final layer: keep continuous for prototype similarity
                layer_input = states

        # Prototype similarity on the final layer's continuous state
        temp = self.log_temp.exp()
        logits = (layer_input @ proto_bp.t()) / temp

        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

model = HDCRWKV(VOCAB_SIZE, D, BLOCK_SIZE, n_layers=N_LAYERS).to(device)
n_params = sum(p.numel() for p in model.parameters())

deploy_bits = VOCAB_SIZE * D + VOCAB_SIZE * D + N_LAYERS * D
print(f'trainable params (continuous): {n_params:,}')
print(f'deployment bits at inference:  {deploy_bits:,} bits = {deploy_bits // 8:,} bytes')
print(f'EEPROM budget 32 KB: {100 * (deploy_bits // 8) / (32*1024):.1f}% used')
print(f'\nArchitecture: {N_LAYERS}-layer recurrent stack with residual + re-binarization')


## Cell 5 — Sanity-check forward pass

At init: random vocab/prototypes mean predictions should be near uniform. Expect loss ≈ ln(128) ≈ 4.85.


In [ ]:
xb, yb = get_batch('train')
with torch.no_grad():
    logits, loss = model(xb, yb)
print(f'logits: {tuple(logits.shape)}')
print(f'init loss: {loss.item():.4f}  (expect ~{math.log(VOCAB_SIZE):.4f})')


## Cell 6 — Training loop with best-val checkpoint

Vanilla cross-entropy. No auxiliary losses. STE handles the bipolar parameter geometry.

### Two metrics we track

- **Soft loss**: forward pass uses STE — parameters appear bipolar at forward, gradients flow through.
- **Hard loss**: forward pass with actual `.sign()` everywhere (no STE) — what the deployment will compute.

A small gap between soft and hard is healthy. A *large* gap means the model relies on continuous slack that doesn't survive binarization.


In [ ]:
@torch.no_grad()
def hard_forward(model, idx, targets):
    """Forward pass with REAL sign — what 6502 actually computes.
    N-layer stacked recurrence with residual + re-binarization between layers.
    """
    B, T = idx.shape
    vocab_bp = model.vocab_hv_c.sign()
    proto_bp = model.prototype_hv_c.sign()

    tok_hv = vocab_bp[idx]
    rotated = torch.zeros_like(tok_hv)
    for t in range(T):
        rotated[:, t, :] = torch.roll(tok_hv[:, t, :], shifts=t, dims=-1)

    layer_input = rotated
    for layer_idx in range(model.n_layers):
        decay_bp = model.decay_masks_c[layer_idx].sign()
        state = torch.zeros(B, model.d, device=idx.device)
        states = []
        for t in range(T):
            update = decay_bp * state + layer_input[:, t, :]
            state = torch.tanh(update)
            states.append(state)
        states = torch.stack(states, dim=1)
        # Residual + re-binarize (real sign for deployment)
        if layer_idx < model.n_layers - 1:
            combined = states + layer_input
            layer_input = combined.sign()
            layer_input[layer_input == 0] = 1.0
        else:
            layer_input = states

    temp = model.log_temp.exp()
    logits = (layer_input @ proto_bp.t()) / temp
    loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
    return logits, loss

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
history = []
best_hard = float('inf')
best_state = None

@torch.no_grad()
def eval_both(n_batches=15):
    model.eval()
    soft_ls = torch.zeros(n_batches)
    hard_ls = torch.zeros(n_batches)
    for k in range(n_batches):
        xb, yb = get_batch('val')
        _, sl = model(xb, yb)
        _, hl = hard_forward(model, xb, yb)
        soft_ls[k] = sl.item()
        hard_ls[k] = hl.item()
    model.train()
    return soft_ls.mean().item(), hard_ls.mean().item()

for step in range(N_STEPS + 1):
    if step % EVAL_EVERY == 0:
        soft_v, hard_v = eval_both()
        history.append((step, soft_v, hard_v))
        marker = ''
        if hard_v < best_hard:
            best_hard = hard_v
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            marker = '  <-- new best (hard)'
        print(f'step {step:>5} | soft val {soft_v:.4f} | hard val {hard_v:.4f}{marker}')
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

print(f'\nbest hard val: {best_hard:.4f}')
model.load_state_dict(best_state)


## Cell 7 — Loss curves: soft vs hard

This plot is THE diagnostic for STE-based binary training. Watch for:

- Both curves should descend together — model is learning.
- The gap between them indicates how well the continuous parameters survive sign quantization. A gap < 0.5 nats means binarization is essentially free.
- If hard loss plateaus much higher than soft loss, the model is relying on continuous slack that doesn't survive deployment. Need to reduce STE 'cheating' (e.g. add KL between sign'd and continuous distributions).


In [ ]:
steps, soft, hard = zip(*history)
plt.figure(figsize=(10, 4.5))
plt.plot(steps, soft, label='soft val (STE-forward)', alpha=0.8)
plt.plot(steps, hard, label='hard val (REAL sign, deployment)', linewidth=2, color='C2')
plt.axhline(3.04, color='red', linestyle='--', alpha=0.7, label='nb07b dense transformer (3.04)')
plt.xlabel('step'); plt.ylabel('val loss (nats/token)')
plt.title(f'HDC-RWKV (d={D}): soft training vs hard deployment loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print(f'\nFinal numbers:')
print(f'  HDC-RWKV soft val:  {soft[-1]:.4f}')
print(f'  HDC-RWKV hard val:  {hard[-1]:.4f}')
print(f'  STE binarization gap: {hard[-1] - soft[-1]:+.4f}  (smaller = cleaner)')
print(f'  vs nb07b transformer (3.04): {hard[-1] - 3.04:+.4f}')


## Cell 8 — Generate text from the binarized model

This uses **hard forward** (real sign, no STE) — what the 6502 will actually execute.

### Expected qualitative profile

- Output stays in the *learned prototype geometry* — should be more semantic than nb11 (random vocab) but less varied than transformer (continuous softmax over full vocab).
- Recurrence captures longer-range patterns than nb11 (T=8 fixed) — should produce more coherent phrase-level structure.
- Still constrained to vocab tokens that have been *learned-good prototypes*, so unlikely to produce token-salad garbage.


In [ ]:
@torch.no_grad()
def generate(prompt='', max_new_tokens=120, temperature=0.6, top_k=5):
    """Generate text from the binarized model.

    top_k filters the distribution to its top-k most likely tokens before sampling.
    Eliminates the long tail of garbage tokens that hurt coherence at small scale.
    Set top_k=0 to disable.
    """
    model.eval()
    ids = encode(prompt) if prompt else [1]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -BLOCK_SIZE:]
        logits, _ = hard_forward(model, idx_cond, idx_cond)
        last_logits = logits[:, -1, :] / temperature
        if top_k > 0:
            # Mask everything outside the top-k
            top_vals, top_idxs = last_logits.topk(top_k, dim=-1)
            mask = torch.full_like(last_logits, float('-inf'))
            mask.scatter_(-1, top_idxs, top_vals)
            last_logits = mask
        probs = F.softmax(last_logits, dim=-1)
        nxt = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

print('--- prompt="king", temp=0.6, top_k=5 ---')
print(generate(prompt='king', max_new_tokens=120, temperature=0.6, top_k=5))
print('\n--- prompt="romeo", temp=0.4, top_k=3 ---')
print(generate(prompt='romeo', max_new_tokens=120, temperature=0.4, top_k=3))
print('\n--- prompt="king", temp=0.8, top_k=10 ---')
print(generate(prompt='king', max_new_tokens=120, temperature=0.8, top_k=10))


## Cell 9 — Inspect what the model learned about decay

The `decay_mask` is `d`-dimensional bipolar. Each channel either preserves state (`+1`) or flips it (`-1`) at every step.

In a useful model we'd expect most channels to preserve (most info is signal), with a minority of flippers (encoding 'novelty' or 'alternation'). A degenerate model would have all channels the same.


In [ ]:
fig, axes = plt.subplots(N_LAYERS, 2, figsize=(10, 3.5 * N_LAYERS))
if N_LAYERS == 1:
    axes = axes.reshape(1, 2)

for layer_idx in range(N_LAYERS):
    decay_continuous = model.decay_masks_c[layer_idx].detach().cpu().numpy()
    decay_binary = np.sign(decay_continuous)

    ax0, ax1 = axes[layer_idx, 0], axes[layer_idx, 1]
    ax0.hist(decay_continuous, bins=40)
    ax0.set_title(f'Layer {layer_idx} — continuous values')
    ax0.set_xlabel('value'); ax0.set_ylabel('# channels')
    ax0.axvline(0, color='red', linestyle='--')

    keep_frac = (decay_binary > 0).mean()
    ax1.bar(['flip (-1)', 'keep (+1)'],
            [(decay_binary < 0).sum(), (decay_binary > 0).sum()],
            color=['C3', 'C0'])
    ax1.set_title(f'Layer {layer_idx} — binary (keep: {keep_frac:.1%})')
    ax1.set_ylabel('# channels')

plt.tight_layout(); plt.show()

# Verdict per layer
print('\nPer-layer decay diagnostic:')
for layer_idx in range(N_LAYERS):
    decay_binary = np.sign(model.decay_masks_c[layer_idx].detach().cpu().numpy())
    keep_frac = (decay_binary > 0).mean()
    if 0.5 <= keep_frac <= 0.95:
        verdict = 'Healthy mix of memory + alternation channels.'
    elif keep_frac > 0.95:
        verdict = 'Degenerate: nearly all channels keep (sums state forever).'
    else:
        verdict = 'Suspicious: more flip than keep — model may not be using memory.'
    print(f'  Layer {layer_idx}: keep={keep_frac:.2%}  flip={1-keep_frac:.2%}  — {verdict}')


## Cell 10 — Compare to all prior architectures

The architectural comparison table for the paper.


In [ ]:
# Hard-coded results from prior notebooks (update with your actual numbers)
comparison = {
    'nb07b transformer (dense)': {'val_nats': 3.04, 'cycles_per_token': 17000*80,  'eeprom_KB': 28.9, 'has_NaN_risk': True,  'integer_only': False},
    'nb09 MoE (sparse)':         {'val_nats': None, 'cycles_per_token': 6000*80,   'eeprom_KB': 28,   'has_NaN_risk': True,  'integer_only': False},
    'nb10 RWKV':                 {'val_nats': None, 'cycles_per_token': 4000*80,   'eeprom_KB': 23,   'has_NaN_risk': True,  'integer_only': False},
    'nb11 HDC (Hebbian)':        {'val_nats': None, 'cycles_per_token': 155000,    'eeprom_KB': 32,   'has_NaN_risk': False, 'integer_only': True},
    'nb12 HDC-RWKV (this)':      {'val_nats': hard[-1], 'cycles_per_token': 135000, 'eeprom_KB': 17, 'has_NaN_risk': False, 'integer_only': True},
}

print(f'{"Architecture":<30} | {"val nats":>9} | {"ms/tok 1MHz":>12} | {"EEPROM KB":>10} | {"int only":>9}')
print('-' * 90)
for arch, d in comparison.items():
    vl = f'{d["val_nats"]:.4f}' if d['val_nats'] is not None else '  TBD'
    print(f'{arch:<30} | {vl:>9} | {d["cycles_per_token"]/1e6*1000:>9.0f} ms | {d["eeprom_KB"]:>10.1f} | {str(d["integer_only"]):>9}')


## Cell 11 — Pack `wozformer_hdcrwkv.bin`

At inference, deployed weights are bipolar. Storage is 1 bit per dimension.

```
offset  size       contents
------  ----       --------
0       4          magic 'WHRK'
4       1          version (1)
5       1          vocab_size (128)
6       2          d / 8 (bytes per vector, LE)
8       1          block_size T
9       4          log_temp (float32, LE) — output softmax temperature
13      3          reserved

16      V·d/8      vocab_hv (packed bits)
...     V·d/8      prototype_hv (packed bits)
...     d/8        decay_mask (packed bits)
```

At d=512: 16 + 8192 + 8192 + 64 = 16,464 bytes = ~16 KB. Half the EEPROM budget. 16 KB of headroom for the BPE tokenizer + program code.


In [ ]:
def pack_bits(continuous_tensor):
    """Sign-binarize then bit-pack. Returns numpy array."""
    binary = (continuous_tensor > 0).to(torch.uint8).cpu().numpy()
    return np.packbits(binary, axis=-1, bitorder='big')

vocab_packed = pack_bits(model.vocab_hv_c.data)
proto_packed = pack_bits(model.prototype_hv_c.data)
# One decay mask per layer
decay_packed_layers = [pack_bits(dm.data.unsqueeze(0)).squeeze(0)
                      for dm in model.decay_masks_c]

export_dir = Path('../export'); export_dir.mkdir(exist_ok=True)
out_path = export_dir / 'wozformer_hdcrwkv.bin'

buf = bytearray()
buf += b'WHRK'
buf += bytes([2, VOCAB_SIZE])           # version=2 (multi-layer)
buf += struct.pack('<H', D // 8)
buf += bytes([BLOCK_SIZE, N_LAYERS])    # block size, num layers
buf += struct.pack('<f', model.log_temp.item())
buf += bytes(2)                          # reserved
buf += vocab_packed.tobytes()
buf += proto_packed.tobytes()
for dp in decay_packed_layers:
    buf += dp.tobytes()

out_path.write_bytes(buf)

BUDGET = 32 * 1024
print(f'wrote {out_path}  ({len(buf):,} bytes)')
print(f'EEPROM budget (32 KB): {100*len(buf)/BUDGET:.1f}% used')
print(f'headroom: {BUDGET - len(buf):,} bytes for BPE table + program')


## Cell 12 — Save Python checkpoint


In [ ]:
ckpt = Path('../export/wozformer_hdcrwkv.pt')
torch.save({
    'config': dict(vocab_size=VOCAB_SIZE, d=D, block_size=BLOCK_SIZE),
    'model_state': model.state_dict(),
    'itos': itos,
    'merges': [(list(p), m) for p, m in merges],
    'best_val_hard': best_hard,
    'history': history,
}, ckpt)
print(f'wrote {ckpt}  ({ckpt.stat().st_size:,} bytes)')


## Post-mortem — the headline architecture

### What's empirically true after running this

- **Hard val loss**: actual deployment-mode loss. Compare to nb07b's 3.04.
- **STE gap**: how much the binarization costs you. Smaller gap = cleaner deployment.
- **Decay mask balance**: did the model learn diverse memory dynamics?
- **Output coherence**: by visual inspection — is generated text comparable to RWKV / better than nb11?

### Three honest outcomes

1. **Hard loss ≤ RWKV's**: HDC-RWKV wins on quality AND on stability/speed. This is the paper's headline result. Ship it.

2. **Hard loss ≈ HDC's (worse than RWKV)**: hybrid didn't outperform the simpler HDC. Either the architecture needs more layers / channel mixing, or the bipolar constraint really is too restrictive at our scale. Paper becomes about *why*, not *that it works best*.

3. **Training unstable / hard gap large**: STE training didn't converge well at this scale. Architectural change needed (e.g. softer quantization during training, slow ramp-up).

All three outcomes are publishable — they're each a contribution about *what works and what doesn't at sub-32 KB scale*.

### The paper now writes itself

Section structure:

1. Introduction — sub-32 KB language models are an unexplored regime.
2. Background — VSA / HDC primitives; RWKV linear recurrence.
3. Architecture — HDC-RWKV; binary recurrence with STE.
4. Experimental setup — 5-way comparison (transformer, MoE, RWKV, HDC, HDC-RWKV) on Tiny Shakespeare.
5. Results — quality, stability, hardware cost.
6. Deployment — 1 MHz 6502 demo (the bombshell).
7. Limitations — single corpus, T=16, no multi-seed yet.
8. Conclusion.

### Onwards

Pick the best generative architecture (likely whichever of nb07b/nb10/nb12 won) + the RAG retrieval (nb08) for the dual-mode 6502 build. C reference + Arduino + 6502 firmware next.
